# NLP-Based OCR Spelling Correction — Pipeline Exploration

This notebook walks through the complete pipeline implemented in `app.py`:

1. Image pre-processing (grayscale → denoise → adaptive threshold)
2. Tesseract OCR text extraction
3. NLTK text tokenisation
4. Language detection
5. LanguageTool language-model correction
6. Accuracy evaluation


## 1. Imports & Setup

In [ ]:
import os
import sys

import cv2
import nltk
import numpy as np
import pytesseract
import language_tool_python
from langdetect import detect, LangDetectException
from nltk.tokenize import sent_tokenize, word_tokenize
from PIL import Image
import matplotlib.pyplot as plt

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Windows path configuration
if sys.platform == 'win32':
    _win = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    if os.path.exists(_win):
        pytesseract.pytesseract.tesseract_cmd = _win

print('Setup complete.')

## 2. Image Pre-Processing Pipeline

Good OCR starts with a clean image. The three-stage pipeline below reliably improves character recognition on noisy scans and photographs of handwritten text.

In [ ]:
def preprocess_image(pil_img: Image.Image) -> Image.Image:
    """Grayscale → denoise → adaptive threshold."""
    arr = np.array(pil_img)

    # Step 1 – grayscale
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY) if arr.ndim == 3 else arr

    # Step 2 – fast non-local means denoising
    denoised = cv2.fastNlMeansDenoising(gray, h=10)

    # Step 3 – adaptive Gaussian thresholding (binarisation)
    binary = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2,
    )
    return Image.fromarray(binary)


def show_pipeline(image_path: str):
    """Visualise each pre-processing stage."""
    original = Image.open(image_path)
    processed = preprocess_image(original)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(original, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(processed, cmap='gray')
    axes[1].set_title('Pre-processed (ready for OCR)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


# Visualise with one of the sample images
show_pipeline('1.jpeg')

## 3. Tesseract OCR Extraction

`--oem 1` selects the LSTM neural-network engine; `--psm 3` lets Tesseract auto-detect page layout.

In [ ]:
OCR_CONFIG = '--oem 1 --psm 3'

def run_ocr(image_path: str) -> str:
    img = Image.open(image_path)
    preprocessed = preprocess_image(img)
    return pytesseract.image_to_string(preprocessed, config=OCR_CONFIG)


raw_text = run_ocr('1.jpeg')
print('=== Raw OCR Output ===')
print(repr(raw_text[:500]))

## 4. NLP Tokenisation with NLTK

Raw OCR output often contains irregular whitespace and broken sentences. NLTK's tokenisers normalise the text before it reaches the language model.

In [ ]:
def tokenize(text: str) -> str:
    """Sentence-segment and word-tokenize, then rejoin cleanly."""
    sentences = sent_tokenize(text)
    return '\n'.join(' '.join(word_tokenize(s)) for s in sentences)


tokenized_text = tokenize(raw_text)
print('=== Tokenised Text (first 400 chars) ===')
print(tokenized_text[:400])

## 5. Language Detection

`langdetect` identifies the dominant language so the correct LanguageTool model is loaded, enabling multilingual document support.

In [ ]:
LANGUAGE_MAP = {
    'en': 'en-US',
    'fr': 'fr',
    'de': 'de-DE',
    'es': 'es',
    'pt': 'pt-PT',
}
DEFAULT_LANG = 'en-US'

def detect_language(text: str) -> str:
    try:
        code = detect(text)
        return LANGUAGE_MAP.get(code, DEFAULT_LANG)
    except LangDetectException:
        return DEFAULT_LANG


lang_code = detect_language(tokenized_text)
print(f'Detected language code: {lang_code}')

## 6. Language-Model Correction with LanguageTool

LanguageTool applies over 5 000 grammar and spelling rules, acting as the language-model layer that cleans up Tesseract's noisy output.

In [ ]:
def apply_language_model(text: str, lang_code: str = DEFAULT_LANG) -> str:
    tool = language_tool_python.LanguageTool(lang_code)
    try:
        return tool.correct(text)
    finally:
        tool.close()


corrected_text = apply_language_model(tokenized_text, lang_code)
print('=== Corrected Text (first 400 chars) ===')
print(corrected_text[:400])

## 7. Accuracy Evaluation

Character-level accuracy is calculated by comparing the corrected output against a ground-truth reference. We use normalised edit distance (Levenshtein).

In [ ]:
def edit_distance(s1: str, s2: str) -> int:
    """Standard dynamic-programming edit distance."""
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            prev, dp[j] = dp[j], (
                prev if s1[i - 1] == s2[j - 1]
                else 1 + min(prev, dp[j], dp[j - 1])
            )
    return dp[n]


def character_accuracy(hypothesis: str, reference: str) -> float:
    if not reference:
        return 0.0
    dist = edit_distance(hypothesis, reference)
    return max(0.0, 1.0 - dist / len(reference))


# Example comparison with a known ground-truth string
ground_truth = 'Welcome to project'   # replace with actual reference
raw_sample   = 'Welcame too procet'
corrected_sample = 'Welcome to project'

acc_before = character_accuracy(raw_sample, ground_truth)
acc_after  = character_accuracy(corrected_sample, ground_truth)
improvement = (
    (acc_after - acc_before) / acc_before * 100
    if acc_before > 0 else float('inf')
)

print(f'Accuracy before correction : {acc_before:.1%}')
print(f'Accuracy after  correction : {acc_after:.1%}')
print(f'Relative improvement       : {improvement:.0f} %')